# Stage1 Elastic SOXL Overflow V1

假设：冻结现有 Stage1，只在其 SOXL 已触及 30% 上限时，把最多 5%/10% 的 SOXX 临时替换为 SOXL，现金不变。先用 2018–2022 expanding walk-forward 判断合理性与 overfit 风险；2023–2026 仅作复用外层诊断，不参与选模。

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
if not (ROOT / 'SPEC.md').exists():
    raise RuntimeError('repository root not found')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CARD = ROOT / 'research' / 'stage1_elastic_soxl_overflow_v1'
PROVIDER = ROOT / 'data' / 'qlib' / 'soxl_soxx_soxs'


In [2]:
qlib.init(provider_uri=str(PROVIDER), region='us', kernels=1)
calendar = D.calendar(start_time='2016-01-01', end_time='2026-12-31', freq='day')
readback = D.features(['SOXL', 'SOXX'], ['$open', '$close'], start_time='2026-07-20', end_time='2026-07-31', freq='day')
{'qlib_version': qlib.__version__, 'calendar_rows': len(calendar), 'calendar_end': str(calendar[-1]), 'readback_rows': len(readback)}


[93684:MainThread](2026-07-27 11:48:56,539) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[93684:MainThread](2026-07-27 11:48:56,540) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[93684:MainThread](2026-07-27 11:48:56,540) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_soxx_soxs')}


{'qlib_version': '0.9.7',
 'calendar_rows': 2654,
 'calendar_end': '2026-07-24 00:00:00',
 'readback_rows': 10}

## Overfit protocol

- 候选固定为 8 个：always/logistic × 5%/10%，Logistic 仅三个固定阈值且 C 固定。
- 仅使用 6 个已有、prior-close 可用特征。
- 2016–2017 仅训练；2018–2022 expanding walk-forward 选模；标签边界 purge 1 行。
- 2023–2026 共用训练截止 2022-12-30 的冻结模型，outer 不得救回开发期失败候选。
- 单独检查样本/特征比、年度稳定性、参数邻域、事件集中度与 outer 事件频率漂移。

In [3]:
from src.workflow.stage1_elastic_soxl_overflow import run_stage1_elastic_soxl_overflow_card

summary = run_stage1_elastic_soxl_overflow_card(root=ROOT)
{'decision': summary['decision'], 'recorder_id': summary['recorder_id'], 'selected_model': summary['selected_model'], 'overfit': summary['overfit_audit']}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[93684:MainThread](2026-07-27 11:48:57,828) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[93684:MainThread](2026-07-27 11:48:57,829) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[93684:MainThread](2026-07-27 11:48:57,830) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_soxx_soxs')}


[93684:MainThread](2026-07-27 11:48:57,852) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[93684:MainThread](2026-07-27 11:48:59,022) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[93684:MainThread](2026-07-27 11:48:59,144) INFO - qlib.workflow - [exp.py:258] - Experiment 313408341576843222 starts running ...


[93684:MainThread](2026-07-27 11:48:59,198) INFO - qlib.workflow - [recorder.py:345] - Recorder 33b59b75c3c040179c8e341f33ceb50f starts running under Experiment 313408341576843222 ...


[93684:MainThread](2026-07-27 11:48:59,277) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'decision': 'not_promoted_development_negative_high_overfit_risk',
 'recorder_id': '33b59b75c3c040179c8e341f33ceb50f',
 'selected_model': {'name': 'always_005',
  'model_type': 'always',
  'overflow_weight': 0.05,
  'probability_threshold': None,
  'selected_from_segment': 'stitched_walk_forward_2018_2022',
  'train_only_years': [2016, 2017],
  'selection_uses_outer_test': False,
  'outer_role': 'reused_diagnostic_only_not_model_selection',
  'source_stage1_frozen': True,
  'source_training_end': '2022-12-30'},
 'overfit': {'risk_level': 'high',
  'failed_checks': ['minimum_events_per_feature',
   'development_all_gates',
   'parameter_neighborhood_retention',
   'outer_cap_event_rate_ratio',
   'clean_unseen_holdout'],
  'outer_cap_event_rate_ratio': 3.608272268360202,
  'minimum_training_cap_events': 40,
  'minimum_events_per_feature': 6.666666666666667,
  'candidate_count': 8,
  'outer_is_selection_input': False,
  'clean_unseen_holdout_available': False}}

In [4]:
display(pd.read_csv(CARD / 'model_selection.csv'))
display(pd.read_csv(CARD / 'overfit_audit.csv'))
display(pd.read_csv(CARD / 'annual_metrics.csv'))
display(pd.read_csv(CARD / 'backtest_benchmark_comparison.csv'))


,name,model_type,overflow_weight,probability_threshold,selection_uses_outer_test,rows,return_pct,max_drawdown_pct,worst_day_pct,cvar5_pct,...,positive_return_gate,action_count_gate,positive_action_rate_gate,positive_years_gate,max_drawdown_gate,worst_5d_gate,event_concentration_gate,gate_pass_count,all_development_gates_pass,selected
0,always_005,always,0.05,NaN,False,1259,220.583837,-27.911568,-7.058978,-4.340540,...,False,True,True,False,True,True,True,5,False,True
1,always_010,always,0.10,NaN,False,1259,219.701327,-28.334349,-7.058978,-4.376264,...,False,True,True,False,True,True,True,5,False,False
2,logistic_055_005,logistic,0.05,0.55,False,1259,218.096442,-27.902440,-7.058978,-4.333126,...,False,True,False,False,True,True,True,4,False,False
3,logistic_055_010,logistic,0.10,0.55,False,1259,214.773076,-28.315365,-7.058978,-4.361436,...,False,True,False,False,True,True,True,4,False,False
4,logistic_065_005,logistic,0.05,0.65,False,1259,218.120091,-27.488582,-7.058978,-4.319508,...,False,True,False,False,True,True,False,3,False,False
5,logistic_065_010,logistic,0.10,0.65,False,1259,214.841950,-27.488582,-7.058978,-4.332733,...,False,True,False,False,True,True,False,3,False,False
6,logistic_075_005,logistic,0.05,0.75,False,1259,219.095204,-27.488582,-7.058978,-4.315215,...,False,False,False,False,True,True,False,2,False,False
7,logistic_075_010,logistic,0.10,0.75,False,1259,216.781670,-27.488582,-7.058978,-4.324146,...,False,False,False,False,True,True,False,2,False,False


,check,value,threshold,passed,interpretation
0,candidate_count,8.000000,<=8,True,small predeclared search grid
1,minimum_training_cap_events,40.000000,>=40,True,earliest expanding fold sample size
2,minimum_events_per_feature,6.666667,>=8.0,False,sparse-event model capacity control
3,development_all_gates,-0.828638,all development gates true,False,selection is not rescued by outer results
4,parameter_neighborhood_retention,0.000000,>=0.50,False,nearby candidate does not collapse
5,outer_cap_event_rate_ratio,3.608272,<=2.0,False,distribution shift between development and outer
6,clean_unseen_holdout,0.000000,required for promotion,False,2023-2026 is reused diagnostic and cannot vali...


,year,segment,rows,cap_bind_count,action_count,baseline_return_pct,sleeve_return_pct,incremental_wealth_pct_points,positive_incremental_year
0,2018,development_walk_forward,251,11,11,-0.927044,-0.589554,0.337490,True
1,2019,development_walk_forward,252,46,46,77.560067,76.800434,-0.759633,False
2,2020,development_walk_forward,253,3,3,58.546423,58.470012,-0.076411,False
3,2021,development_walk_forward,252,4,4,49.892375,50.589150,0.696775,True
4,2022,development_walk_forward,251,5,5,-23.117709,-23.566191,-0.448482,False
5,2023,outer_reused_diagnostic,250,14,14,29.532618,28.962359,-0.570259,False
6,2024,outer_reused_diagnostic,252,5,5,52.616368,52.645924,0.029556,True
7,2025,outer_reused_diagnostic,250,77,77,61.937432,64.951050,3.013618,True
8,2026,outer_reused_diagnostic,138,80,80,100.747085,105.918256,5.171171,True


,segment,rows,return_pct,max_drawdown_pct,worst_day_pct,cvar5_pct,worst_rolling_5d_pct,annualized_volatility_pct,baseline_return_pct,baseline_max_drawdown_pct,baseline_worst_rolling_5d_pct,return_delta_pct_points,max_drawdown_delta_pct_points,worst_5d_delta_pct_points,action_count,cap_bind_count,cap_bind_rate,positive_action_rate,incremental_turnover,top5_positive_contribution_share
0,full_2016_2022_context_includes_train_only_not...,1762,415.765183,-27.911568,-7.845988,-4.160126,-11.858940,27.231281,417.272054,-27.488582,-11.858940,-1.506871,-4.229855e-01,0.000000,110,110,0.062429,0.572727,2.4,0.239562
1,development_2018_2022,1259,220.583837,-27.911568,-7.058978,-4.340540,-11.858940,29.496602,221.412475,-27.488582,-11.858940,-0.828638,-4.229855e-01,0.000000,69,69,0.054805,0.579710,1.3,0.328449
2,outer_reused_diagnostic_2023_2026,890,568.648873,-28.105112,-10.086541,-5.349024,-12.243653,37.872862,542.653315,-28.105112,-11.602809,25.995557,-1.065814e-14,-0.640844,176,176,0.197753,0.579545,1.3,0.149353
3,outer_reused_diagnostic_25bps,890,495.238054,-29.088722,-10.086541,-5.360404,-12.426677,37.868926,473.196179,-29.088722,-11.780256,22.041876,1.065814e-14,-0.646421,176,176,0.197753,0.579545,1.3,0.149353


## 中文结论

最终结论必须覆盖：做了什么、开发期与 outer 指标、年度/事件稳定性、overfit 风险、是否晋升、风险和下一步。下面直接显示与 recorder/sidecar 同步的 `conclusion.md`。

In [5]:
from IPython.display import Markdown, display
display(Markdown((CARD / 'conclusion.md').read_text(encoding='utf-8')))

# 中文结论

- 做了什么：冻结当前 Stage1，只在其 SOXL 已触及 30% 时测试额外 5%/10% overflow；资金只从 SOXX 调拨，现金权重不变，信号使用 prior-close 特征并在 next-open 生效。
- 过拟合控制：只搜索 8 个预先固定候选；Logistic C 固定为 0.10，特征固定为 6 个；2016–2017 仅训练，2018–2022 expanding walk-forward 选模，2023–2026 只诊断且所有 outer 年份共用训练截止 2022-12-30 的冻结模型。
- 选中诊断候选：always_005；开发期增量收益=-0.8286 个百分点，动作=69 次，正增量年份=2/5。
- 整段稳健性检查：若把 2016–2017 训练期也并入整段，增量收益=-1.5069 个百分点；该数字同样为负，说明当前 Stage1 口径下并不存在被训练期掩盖的稳定 overflow 收益。
- 开发期风险：MaxDD 变化=-0.4230 个百分点，最差 5 日变化=0.0000 个百分点，正动作比例=57.97%。
- 外层复用诊断：增量收益=25.9956 个百分点，动作=176 次；该结果不得反向选择参数。
- overfit 风险=high；失败项=['minimum_events_per_feature', 'development_all_gates', 'parameter_neighborhood_retention', 'outer_cap_event_rate_ratio', 'clean_unseen_holdout']。主要问题是开发期没有正增量、早期 fold 每个特征对应事件数不足、outer 触顶频率相对开发期发生明显漂移，而且没有干净 holdout。
- 决策：not_promoted_development_negative_high_overfit_risk；不修改生产 Stage1，不开启 overflow paper/sim。只有新增独立 PIT 信息或预登记 clean-forward 后，才值得重开。
- qlib recorder：33b59b75c3c040179c8e341f33ceb50f
- 不构成买卖建议。
